# UTS — RAG Literature Gap Matrix



In [1]:
# 1. Instalasi dan import
!pip -q install pymupdf scikit-learn pandas

import os, re, glob, json, textwrap, hashlib, zipfile
import fitz
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import files
from IPython.display import display

print("Library siap.")

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Library siap.


In [2]:
# 2. Hubungkan Google Drive dan tentukan folder
from google.colab import drive
drive.mount('/content/drive')

PDF_DIR = "/content/drive/MyDrive/literature_review"
OUT_DIR = "/content/drive/MyDrive/literature_review_output"
os.makedirs(OUT_DIR, exist_ok=True)

pdf_files = sorted(glob.glob(os.path.join(PDF_DIR, "*.pdf")))
print("Folder PDF:", PDF_DIR)
print("Folder output:", OUT_DIR)
print("Jumlah PDF:", len(pdf_files))

if not os.path.isdir(PDF_DIR):
    raise FileNotFoundError(
        f"Folder {PDF_DIR} belum ditemukan. Buat folder 'literature_review' "
        "di MyDrive atau ubah PDF_DIR sesuai lokasi folder kamu."
    )

if not pdf_files:
    raise FileNotFoundError(
        f"Tidak ada PDF di {PDF_DIR}. Pastikan folder 'literature_review' ada di MyDrive, "
        "atau ubah PDF_DIR sesuai folder kamu."
    )

for f in pdf_files:
    print("-", os.path.basename(f))

if len(pdf_files) < 20:
    print(f"PERINGATAN: baru ditemukan {len(pdf_files)} PDF. Jika tugas meminta 20 artikel, tambahkan PDF yang belum ada.")
elif len(pdf_files) > 20:
    print(f"PERINGATAN: ditemukan {len(pdf_files)} PDF. Pastikan semuanya termasuk artikel yang relevan.")

Mounted at /content/drive
Folder PDF: /content/drive/MyDrive/literature_review
Folder output: /content/drive/MyDrive/literature_review_output
Jumlah PDF: 10
- A Review of Supply Chain Risk Management: Definition, Theory, and Research Agenda.pdf
- Achieving competitive advantage through technology-driven proactive supply chain risk management: an empirical study.pdf
- Artificial intelligence applications for supply chain risk management considering interconnectivity, external events exposures and transparency: a systematic literature review.pdf
- Dealing with supply chain risks: Linking risk management practices and strategies to performance.pdf
- Enhancing Logistics Performance through Increased Trust and Collaboration in Supply Chain Risk Management: A Focus on the Distribution Network of Manufacturing Companies.pdf
- Supply Chain Risk Management: A Content Analysis Approach.pdf
- Supply Chain Risk Management: Literature Review_Amulya.pdf
- Supply chain risk management: A content anal

In [3]:
# 3. Ekstraksi teks per halaman; halaman dipertahankan untuk bukti kutipan
articles = []
errors = []

def normalize_text(s):
    s = s.replace("\u00ad", "")
    s = re.sub(r'-\s*\n\s*', '', s)   # sambung kata terpotong di akhir baris
    s = re.sub(r'\s+', ' ', s)
    return s.strip()

for path in pdf_files:
    try:
        page_records = []
        with fitz.open(path) as doc:
            page_count = len(doc)
            for page_idx, page in enumerate(doc):
                raw = page.get_text("text")
                clean = normalize_text(raw)
                if clean:
                    page_records.append({"page": page_idx + 1, "text": clean})
        full_text = "\n".join(p["text"] for p in page_records)
        articles.append({
            "filename": os.path.basename(path),
            "path": path,
            "pages": page_count,
            "page_records": page_records,
            "text": full_text
        })
        print(f"{os.path.basename(path)} | halaman={page_count} | karakter={len(full_text)}")
        if len(full_text) < 300:
            errors.append((os.path.basename(path), "Teks sangat sedikit/kosong; kemungkinan PDF hasil scan dan perlu OCR."))
    except Exception as e:
        errors.append((os.path.basename(path), f"Gagal dibaca: {e}"))

print("\nPDF berhasil dibuka:", len(articles))
if errors:
    print("\nPeringatan:")
    for name, err in errors:
        print("-", name, ":", err)
if not articles:
    raise ValueError("Tidak ada PDF yang berhasil dibuka.")

A Review of Supply Chain Risk Management: Definition, Theory, and Research Agenda.pdf | halaman=36 | karakter=83668
Achieving competitive advantage through technology-driven proactive supply chain risk management: an empirical study.pdf | halaman=42 | karakter=138192
Artificial intelligence applications for supply chain risk management considering interconnectivity, external events exposures and transparency: a systematic literature review.pdf | halaman=32 | karakter=122622
Dealing with supply chain risks: Linking risk management practices and strategies to performance.pdf | halaman=23 | karakter=75696
Enhancing Logistics Performance through Increased Trust and Collaboration in Supply Chain Risk Management: A Focus on the Distribution Network of Manufacturing Companies.pdf | halaman=18 | karakter=67519
Supply Chain Risk Management: A Content Analysis Approach.pdf | halaman=9 | karakter=50807
Supply Chain Risk Management: Literature Review_Amulya.pdf | halaman=16 | karakter=59107
Supply

In [4]:
# 4. Chunking dengan metadata halaman
CHUNK_SIZE = 1200
OVERLAP = 200
chunks = []

for article in articles:
    for page_rec in article["page_records"]:
        text = page_rec["text"]
        start = 0
        while start < len(text):
            piece = text[start:start + CHUNK_SIZE].strip()
            if piece:
                chunks.append({
                    "chunk_id": len(chunks),
                    "filename": article["filename"],
                    "page": page_rec["page"],
                    "text": piece
                })
            start += max(1, CHUNK_SIZE - OVERLAP)

print("Jumlah chunks:", len(chunks))
if not chunks:
    raise ValueError("Tidak ada teks yang bisa di-chunk. Periksa apakah PDF berbasis teks atau perlu OCR.")

Jumlah chunks: 1068


In [5]:
# 5. Indeks pencarian lokal TF-IDF (tanpa API key)
chunk_texts = [c["text"] for c in chunks]
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    max_features=100000,
    sublinear_tf=True
)
tfidf_matrix = vectorizer.fit_transform(chunk_texts)
print("Indeks TF-IDF:", tfidf_matrix.shape)

def retrieve(query, filename=None, top_k=5):
    qv = vectorizer.transform([query])
    scores = cosine_similarity(qv, tfidf_matrix).ravel()
    candidates = list(range(len(chunks)))
    if filename:
        candidates = [i for i in candidates if chunks[i]["filename"] == filename]
    ranked = sorted(candidates, key=lambda i: scores[i], reverse=True)[:top_k]
    out = []
    for i in ranked:
        if scores[i] > 0:
            out.append({"score": float(scores[i]), **chunks[i]})
    return out

# Tes retrieval
for result in retrieve("supply chain risk management limitations future research", top_k=3):
    print(f"\n{result['filename']} | halaman {result['page']} | skor {result['score']:.4f}")
    print(result["text"][:400], "...")

Indeks TF-IDF: (1068, 77718)

Dealing with supply chain risks: Linking risk management practices and strategies to performance.pdf | halaman 15 | skor 0.1363
sk management: understanding the business requirements from a practitioner perspective”, The International Journal of Logistics Management, Vol. 16 No. 1, pp. 120-41. Ju¨ttner, U., Peck, H. and Christopher, M. (2003), “Supply chain risk management: outlining an agenda for future research”, International Journal of Logistics: Research and Applications, Vol. 6 No. 4, pp. 197-210. Dealing with supply ...

Supply chain risk management: A content analysis-based review of existing and emerging topics.pdf | halaman 3 | skor 0.1351
Supply Chain Analytics 3 (2023) 100031 3 Table 1 Published Review Papers on SCRs area between 2004 and 2022. Author Year Title Journal Focus Reviewed Papers Time Period Bibliometric View Citations [37] 2004 A Review of Enterprise Supply Chain Risk Management Journal of Systems Science and Systems Engineering En

In [6]:
# 6. Fungsi bantu: kandidat metadata dan kalimat berdasarkan kata kunci
def get_sentences(text):
    text = re.sub(r'\s+', ' ', text)
    parts = re.split(r'(?<=[.!?])\s+(?=[A-Z0-9])', text)
    return [p.strip() for p in parts if len(p.strip()) >= 40]

def first_match(text, patterns, default="Perlu diperiksa manual"):
    for pattern in patterns:
        m = re.search(pattern, text, flags=re.I | re.M)
        if m:
            return m.group(1).strip(" \n\t:;,.")
    return default

def metadata_guess(article):
    text = article["text"]
    front = "\n".join(p["text"] for p in article["page_records"][:2])
    lines = [x.strip() for x in front.splitlines() if x.strip()]
    # Heuristik sederhana: judul kandidat biasanya salah satu baris awal yang cukup panjang
    title = next((x for x in lines if 20 <= len(x) <= 240 and not re.search(r'^(abstract|keywords|introduction|doi|issn)\b', x, re.I)), article["filename"])
    year = first_match(front, [r'\b(20(?:1\d|2\d)|19\d\d)\b'], "Perlu diperiksa manual")
    doi = first_match(text[:8000], [r'(10\.\d{4,9}/[^\s<>\]]+)'], "Tidak terdeteksi otomatis")
    doi = doi.rstrip(".,;)")
    # Penulis tidak ditebak dengan kepastian karena tata letak PDF bervariasi
    return {"Judul kandidat": title, "Tahun kandidat": year, "DOI kandidat": doi,
            "Penulis": "Perlu diperiksa manual", "Nama jurnal": "Perlu diperiksa manual"}

FIELD_TERMS = {
    "Tujuan kandidat": ["objective", "aim of this study", "purpose of this study", "research objective", "the aim of", "this study aims", "tujuan penelitian"],
    "Metode kandidat": ["methodology", "methods", "systematic literature review", "case study", "survey", "we propose", "we develop", "data were collected", "research design", "metode penelitian"],
    "Dataset/konteks kandidat": ["dataset", "data set", "sample", "participants", "case study", "industry", "manufacturing", "logistics", "supplier", "supply chain", "data were collected", "study context"],
    "Temuan kandidat": ["results show", "findings show", "our findings", "results indicate", "we found", "the results suggest", "this study found", "temuan menunjukkan"],
    "Keterbatasan/gap kandidat": ["limitation", "limitations", "future research", "future studies", "further research", "research gap", "remains unclear", "has not been", "little research", "limited research", "lack of", "not yet explored", "keterbatasan", "penelitian selanjutnya"]
}

def extract_candidates(text, terms, max_items=3):
    sentences = get_sentences(text)
    ranked = []
    for s in sentences:
        low = s.lower()
        hits = sum(term in low for term in terms)
        if hits:
            ranked.append((hits, s))
    ranked.sort(key=lambda x: (-x[0], len(x[1])))
    out, seen = [], set()
    for _, s in ranked:
        key = s.lower()
        if key not in seen:
            out.append(s)
            seen.add(key)
        if len(out) >= max_items:
            break
    return " ".join(out) if out else "Tidak ditemukan otomatis; periksa artikel manual."

print("Fungsi metadata dan ekstraksi kandidat siap.")

Fungsi metadata dan ekstraksi kandidat siap.


In [7]:
# 7. Bangun Literature Gap Matrix
rows = []
evidence_rows = []

queries = {
    "Tujuan kandidat": "research objective aim purpose aims of this study",
    "Metode kandidat": "method methodology systematic review survey case study model",
    "Dataset/konteks kandidat": "dataset sample data industry logistics supply chain context",
    "Temuan kandidat": "results findings show indicate main findings",
    "Keterbatasan/gap kandidat": "limitations future research research gap further studies lack of"
}

for article in articles:
    meta = metadata_guess(article)
    # Gabungkan beberapa chunk teratas dari kueri berbeda untuk bahan ekstraksi.
    retrieved_by_field = {}
    for field, query in queries.items():
        found = retrieve(query, filename=article["filename"], top_k=5)
        retrieved_by_field[field] = found
        for rank, item in enumerate(found, start=1):
            evidence_rows.append({
                "File": article["filename"],
                "Bidang pencarian": field,
                "Peringkat": rank,
                "Halaman": item["page"],
                "Skor TF-IDF": round(item["score"], 5),
                "Kutipan konteks": item["text"][:1200]
            })

    # Gunakan seluruh teks artikel untuk ekstraksi berbasis kata kunci, bukan hanya abstrak.
    row = {
        "File": article["filename"],
        "Judul kandidat": meta["Judul kandidat"],
        "Penulis": meta["Penulis"],
        "Tahun kandidat": meta["Tahun kandidat"],
        "Nama jurnal": meta["Nama jurnal"],
        "DOI kandidat": meta["DOI kandidat"],
        "Tujuan kandidat": extract_candidates(article["text"], FIELD_TERMS["Tujuan kandidat"]),
        "Metode kandidat": extract_candidates(article["text"], FIELD_TERMS["Metode kandidat"]),
        "Dataset/konteks kandidat": extract_candidates(article["text"], FIELD_TERMS["Dataset/konteks kandidat"]),
        "Temuan kandidat": extract_candidates(article["text"], FIELD_TERMS["Temuan kandidat"]),
        "Keterbatasan/gap kandidat": extract_candidates(article["text"], FIELD_TERMS["Keterbatasan/gap kandidat"]),
        "Status validasi": "Draft berbasis kata kunci — wajib verifikasi PDF asli"
    }
    rows.append(row)

gap_matrix = pd.DataFrame(rows)
display(gap_matrix.head(10))
print("Jumlah artikel dalam matriks:", len(gap_matrix))

,File,Judul kandidat,Penulis,Tahun kandidat,Nama jurnal,DOI kandidat,Tujuan kandidat,Metode kandidat,Dataset/konteks kandidat,Temuan kandidat,Keterbatasan/gap kandidat,Status validasi
0,A Review of Supply Chain Risk Management: Defi...,A Review of Supply Chain Risk Management: Defi...,Perlu diperiksa manual,Perlu diperiksa manual,Perlu diperiksa manual,Tidak terdeteksi otomatis,The objectives of this review are to: 1. Objec...,Papers % of Total Papers Single Research Metho...,"Lin, Y. & Zhou, L., (2011), “The impacts of pr...",Tidak ditemukan otomatis; periksa artikel manual.,Limitations Our analysis is limited by the ava...,Draft berbasis kata kunci — wajib verifikasi P...
1,Achieving competitive advantage through techno...,Achieving competitive advantage through techno...,Perlu diperiksa manual,2024,Perlu diperiksa manual,10.1007/s10479-023-05604-y,A multi-objective approach to supply chain ris...,"4 we set out our research design, outlining ou...",123 174 Annals of Operations Research (2024) 3...,"First, we checked the factor loadings, with th...",We then highlight the managerial implications ...,Draft berbasis kata kunci — wajib verifikasi P...
2,Artificial intelligence applications for suppl...,Artificial intelligence applications for suppl...,Perlu diperiksa manual,2024,Perlu diperiksa manual,10.1108/MSCRA-10-2024-0041,They recommended considering other objectives ...,"Design/methodology/approach – In the current, ...","Brintrup, A., Pak, J., Ratiney, D., Pearce, T....",The results showed 95% precision for different...,Sections 4 and 5 discuss research gaps and fut...,Draft berbasis kata kunci — wajib verifikasi P...
3,Dealing with supply chain risks: Linking risk ...,Dealing with supply chain risks: Linking risk ...,Perlu diperiksa manual,2011,Perlu diperiksa manual,10.1108/09600031211281411,"Second, except for control variables, no objec...",Utilized multi-method research design Prior th...,"Khan, O., Christopher, M. and Burnes, B. (2008...",Tidak ditemukan otomatis; periksa artikel manual.,This is a possible starting point for further ...,Draft berbasis kata kunci — wajib verifikasi P...
4,Enhancing Logistics Performance through Increa...,Enhancing Logistics Performance through Increa...,Perlu diperiksa manual,2024,Perlu diperiksa manual,10.3390/systems12040141,"On one hand, trust is interpreted differently ...",A Survey of Supply Chain Collaboration and Man...,Article Sample Size Industry Sector Results Co...,"Second, the results showed that affective trus...",Section 4 presents the research methods and re...,Draft berbasis kata kunci — wajib verifikasi P...
5,Supply Chain Risk Management: A Content Analys...,Supply Chain Risk Management: A Content Analys...,Perlu diperiksa manual,2013,Perlu diperiksa manual,Tidak terdeteksi otomatis,The aim of SC risk management is to reduce the...,"828-838. [29] Colicchia C., and Strozzi F. (20...","39 – 51. [66] Lin, Y., and Zhou, L. (2011), “T...",The results show that the proposed approach al...,In accordance with recommendations for initial...,Draft berbasis kata kunci — wajib verifikasi P...
6,Supply Chain Risk Management: Literature Revie...,Supply Chain Risk Management: Literature Revie...,Perlu diperiksa manual,2021,Perlu diperiksa manual,10.3390/risks9010016,"Therefore, quick recovery is the prime objecti...",Supply chain risk management: A new methodolog...,Supply chain optimization under risk and uncer...,Tidak ditemukan otomatis; periksa artikel manual.,This study is not devoid of limitations. There...,Draft berbasis kata kunci — wajib verifikasi P...
7,Supply chain risk management: A content analys...,Supply chain risk management: A content analys...,Perlu diperiksa manual,2023,Perlu diperiksa manual,10.1016/j.sca.2023.100031,"In the lack of excellent corporate culture, ac...","Strozzi, Supply chain risk management: a new m...",A review Advanced Science Letters General defi...,Tidak ditemukan otomatis; periksa artikel manual.,"However, there are still limitations in th

Jumlah artikel dalam matriks: 10


In [8]:
# 8. Simpan matriks dan bukti retrieval beserta halaman
matrix_path = os.path.join(OUT_DIR, "Literature_Gap_Matrix.csv")
evidence_path = os.path.join(OUT_DIR, "Bukti_Kutipan_dan_Halaman.csv")

gap_matrix.to_csv(matrix_path, index=False, encoding="utf-8-sig")
evidence_df = pd.DataFrame(evidence_rows)
evidence_df.to_csv(evidence_path, index=False, encoding="utf-8-sig")

print("Matriks:", matrix_path)
print("Bukti retrieval:", evidence_path)
print("Jumlah baris bukti:", len(evidence_df))

Matriks: /content/drive/MyDrive/literature_review_output/Literature_Gap_Matrix.csv
Bukti retrieval: /content/drive/MyDrive/literature_review_output/Bukti_Kutipan_dan_Halaman.csv
Jumlah baris bukti: 250


In [9]:
# Buat template penilaian manual untuk setiap hasil retrieval
validation_template = evidence_df.copy()
validation_template["Relevansi manual (0/1/2)"] = ""
validation_template["Kutipan mendukung klaim? (Ya/Tidak/Parsial)"] = ""
validation_template["Halaman dicek di PDF asli? (Ya/Tidak)"] = ""
validation_template["Jenis bukti (Eksplisit/Sintesis/Tidak ditemukan)"] = ""
validation_template["Catatan validator"] = ""

validation_path = os.path.join(OUT_DIR, "Template_Validasi_Manual.csv")
validation_template.to_csv(validation_path, index=False, encoding="utf-8-sig")

# Audit sederhana: cek metadata kosong dan panjang kandidat gap
audit_rows = []
for _, row in gap_matrix.iterrows():
    gap_text = str(row.get("Keterbatasan/gap kandidat", "") or "").strip()
    audit_rows.append({
        "File": row.get("File", ""),
        "Judul kandidat tersedia": bool(str(row.get("Judul kandidat", "") or "").strip()),
        "Penulis tersedia": bool(str(row.get("Penulis", "") or "").strip()),
        "Tahun kandidat tersedia": bool(str(row.get("Tahun kandidat", "") or "").strip()),
        "DOI kandidat tersedia": bool(str(row.get("DOI kandidat", "") or "").strip()),
        "Kandidat gap ditemukan": bool(gap_text and gap_text.lower() not in ["nan", "tidak ditemukan"]),
        "Panjang teks kandidat gap": len(gap_text),
        "Status": "Perlu cek PDF asli"
    })
audit_df = pd.DataFrame(audit_rows)
audit_path = os.path.join(OUT_DIR, "Audit_Metadata_dan_Gap.csv")
audit_df.to_csv(audit_path, index=False, encoding="utf-8-sig")

print("Template validasi manual:", validation_path)
print("Audit metadata/gap:", audit_path)
display(validation_template.head(5))
display(audit_df)

Template validasi manual: /content/drive/MyDrive/literature_review_output/Template_Validasi_Manual.csv
Audit metadata/gap: /content/drive/MyDrive/literature_review_output/Audit_Metadata_dan_Gap.csv


,File,Bidang pencarian,Peringkat,Halaman,Skor TF-IDF,Kutipan konteks,Relevansi manual (0/1/2),Kutipan mendukung klaim? (Ya/Tidak/Parsial),Halaman dicek di PDF asli? (Ya/Tidak),Jenis bukti (Eksplisit/Sintesis/Tidak ditemukan),Catatan validator
0,A Review of Supply Chain Risk Management: Defi...,Tujuan kandidat,1,6,0.06787,"artners, and the internal implementation of SC...",,,,,
1,A Review of Supply Chain Risk Management: Defi...,Tujuan kandidat,2,35,0.04431,Figure 2: Objective-Process-Outcome (OPO) Base...,,,,,
2,A Review of Supply Chain Risk Management: Defi...,Tujuan kandidat,3,21,0.04269,al. (2015) respond to Ghadge et al.’s (2012) c...,,,,,
3,A Review of Supply Chain Risk Management: Defi...,Tujuan kandidat,4,4,0.03963,4 none of these reviews have given particular ...,,,,,
4,A Review of Supply Chain Risk Management: Defi...,Tujuan kandidat,5,17,0.03889,"e explicitly discussed. Arguably, research at ...",,,,,


,File,Judul kandidat tersedia,Penulis tersedia,Tahun kandidat tersedia,DOI kandidat tersedia,Kandidat gap ditemukan,Panjang teks kandidat gap,Status
0,A Review of Supply Chain Risk Management: Defi...,True,True,True,True,True,1312,Perlu cek PDF asli
1,Achieving competitive advantage through techno...,True,True,True,True,True,419,Perlu cek PDF asli
2,Artificial intelligence applications for suppl...,True,True,True,True,True,321,Perlu cek PDF asli
3,Dealing with supply chain risks: Linking risk ...,True,True,True,True,True,344,Perlu cek PDF asli
4,Enhancing Logistics Performance through Increa...,True,True,True,True,True,366,Perlu cek PDF asli
5,Supply Chain Risk Management: A Content Analys...,True,True,True,True,True,517,Perlu cek PDF asli
6,Supply Chain Risk Management: Literature Revie...,True,True,True,True,True,157,Perlu cek PDF asli
7,Supply chain risk management: A content analys...,True,True,True,True,True,324,Perlu cek PDF asli
8,Supply chain risk management: A literature rev...,True,True,True,True,True,368,Perlu cek PDF asli
9,Towards knowledge graph reasoning for supply c...,True,True,True,True,True,343,Perlu cek PDF asli


In [10]:
# 9. Sintesis awal lintas artikel (heuristik; bukan kesimpulan otomatis final)
# Kelompokkan kandidat gap menurut istilah yang muncul dalam teks.
all_gap_text = " ".join(gap_matrix["Keterbatasan/gap kandidat"].astype(str)).lower()
gap_themes = {
    "Future research / studi lanjutan": ["future research", "future studies", "further research"],
    "Keterbatasan": ["limitation", "limitations"],
    "Kekurangan bukti/riset terbatas": ["lack of", "limited research", "little research"],
    "Gap eksplisit": ["research gap", "has not been", "not yet explored", "remains unclear"],
    "Konteks Indonesia/lokal": ["indonesia", "indonesian", "local context", "emerging economies"],
    "Teknologi/digital": ["artificial intelligence", "machine learning", "blockchain", "digital", "generative ai"]
}
theme_counts = []
for theme, terms in gap_themes.items():
    count = sum(all_gap_text.count(term) for term in terms)
    theme_counts.append({"Tema kandidat": theme, "Jumlah kemunculan istilah": count,
                         "Catatan": "Frekuensi istilah saja; cek kutipan dan konteks artikel"})
theme_df = pd.DataFrame(theme_counts).sort_values("Jumlah kemunculan istilah", ascending=False)
display(theme_df)

theme_path = os.path.join(OUT_DIR, "Ringkasan_Tema_Gap.csv")
theme_df.to_csv(theme_path, index=False, encoding="utf-8-sig")

# Tulis laporan awal dengan batasan yang jelas
report_path = os.path.join(OUT_DIR, "Analisis_Research_Gap.md")
lines = [
    "# Analisis Awal Research Gap — Supply Chain Risk Management",
    "",
    f"Jumlah PDF yang diproses: {len(articles)}",
    "",
    "## Catatan metode",
    "Analisis ini dibuat tanpa API key menggunakan TF-IDF, cosine similarity, dan pencocokan kata kunci. Frekuensi istilah bukan bukti langsung adanya research gap. Setiap klaim harus diperiksa pada PDF dan bukti halaman.",
    "",
    "## Tema kandidat yang perlu diverifikasi",
]
for _, r in theme_df.iterrows():
    lines.append(f"- **{r['Tema kandidat']}**: {r['Jumlah kemunculan istilah']} kemunculan istilah. {r['Catatan']}.")
lines += [
    "",
    "## Langkah sintesis manual",
    "1. Baca kolom keterbatasan/gap kandidat dan buka kutipan sumber pada file Bukti_Kutipan_dan_Halaman.csv.",
    "2. Kelompokkan gap yang serupa, pisahkan gap metodologis, konteks industri, data, dan teknologi.",
    "3. Bandingkan apakah gap benar-benar belum dijawab oleh artikel lain dalam korpus.",
    "4. Tulis gap akhir dengan menyebutkan artikel dan halaman pendukung.",
    "",
    "## Kerangka paragraf",
    "Literatur yang ditinjau menunjukkan beberapa fokus dalam manajemen risiko rantai pasok, antara lain [tema yang tervalidasi]. Namun, berdasarkan perbandingan artikel [cantumkan sumber dan halaman], masih terdapat keterbatasan terkait [gap yang didukung bukti]. Gap ini penting karena [alasan berdasarkan konteks industri]. Penelitian yang diusulkan akan mengkaji [fokus], dengan kontribusi berupa [kontribusi yang dapat diuji].",
]
with open(report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print("Laporan awal:", report_path)
print("Ringkasan tema:", theme_path)

,Tema kandidat,Jumlah kemunculan istilah,Catatan
1,Keterbatasan,35,Frekuensi istilah saja; cek kutipan dan kontek...
0,Future research / studi lanjutan,9,Frekuensi istilah saja; cek kutipan dan kontek...
2,Kekurangan bukti/riset terbatas,7,Frekuensi istilah saja; cek kutipan dan kontek...
3,Gap eksplisit,5,Frekuensi istilah saja; cek kutipan dan kontek...
4,Konteks Indonesia/lokal,0,Frekuensi istilah saja; cek kutipan dan kontek...
5,Teknologi/digital,0,Frekuensi istilah saja; cek kutipan dan kontek...


Laporan awal: /content/drive/MyDrive/literature_review_output/Analisis_Research_Gap.md
Ringkasan tema: /content/drive/MyDrive/literature_review_output/Ringkasan_Tema_Gap.csv


In [11]:
# 10. Kerangka usulan topik paper (perlu diisi setelah validasi matriks)
proposal_path = os.path.join(OUT_DIR, "Usulan_Topik_Paper.md")
proposal = """# Kerangka Usulan Topik Paper — Supply Chain Risk Management

> Isi bagian dalam tanda kurung setelah memvalidasi 10 artikel dan bukti kutipan.

## Judul sementara
[Topik spesifik yang muncul dari gap terverifikasi]

## Latar belakang
[Jelaskan masalah supply chain yang nyata dan dukung dengan artikel serta halaman.]

## Research problem
[Masalah yang belum terjawab berdasarkan perbandingan beberapa artikel.]

## Research question
1. [RQ1 yang dapat dijawab dengan data/metode yang tersedia]
2. [RQ2 jika memang diperlukan]

## Research gap
- Gap 1: [pernyataan gap] — bukti: [artikel, tahun, halaman].
- Gap 2: [pernyataan gap] — bukti: [artikel, tahun, halaman].
- Gap 3: [pernyataan gap] — bukti: [artikel, tahun, halaman].

## Metode yang diusulkan
[Jelaskan dataset, variabel, metode analisis, pembanding/baseline, dan cara evaluasi.]

## Kontribusi yang diharapkan
[Kontribusi teoretis/metodologis/praktis; jangan klaim novelty sebelum verifikasi literatur.]

## Validasi sebelum dikumpulkan
- [ ] Judul, penulis, tahun, jurnal, DOI diverifikasi.
- [ ] Setiap gap didukung lebih dari satu bukti jika memungkinkan.
- [ ] Gap dibandingkan dengan semua artikel yang relevan.
- [ ] Metode dapat dijalankan dengan data dan sumber daya yang tersedia.
- [ ] Klaim kebaruan tidak berlebihan.
"""
with open(proposal_path, "w", encoding="utf-8") as f:
    f.write(proposal)
print("Kerangka usulan paper:", proposal_path)

Kerangka usulan paper: /content/drive/MyDrive/literature_review_output/Usulan_Topik_Paper.md


In [12]:
# 11. Opsional: muat model bahasa lokal tanpa API key
# Model akan diunduh saat pertama kali dijalankan; tidak memerlukan API key.
!pip -q install transformers sentencepiece

LOCAL_LLM_AVAILABLE = False
local_generator = None

try:
    import torch
    from transformers import pipeline
    device = 0 if torch.cuda.is_available() else -1
    local_generator = pipeline(
        "text2text-generation",
        model="google/flan-t5-small",
        device=device
    )
    LOCAL_LLM_AVAILABLE = True
    print("Model lokal siap. Device:", "GPU" if device == 0 else "CPU")
except Exception as e:
    print("Model lokal tidak berhasil dimuat.")
    print("Matriks dasar tetap tersedia. Detail:", repr(e))

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

Model lokal tidak berhasil dimuat.
Matriks dasar tetap tersedia. Detail: KeyError('The task text2text-generation has been removed from transformers. Use the `text-generation` pipeline with a chat model instead. See the migration guide for more details: https://github.com/huggingface/transformers/blob/main/MIGRATION_GUIDE_V5.md#pipelines')


In [13]:
# 12. Generate sintesis research gap dengan konteks hasil retrieval
# Menghasilkan draft berbasis konteks; bukan pengganti validasi manual.
if LOCAL_LLM_AVAILABLE:
    query = (
        "supply chain risk management research gaps limitations future research "
        "digital technology resilience risk mitigation"
    )
    retrieved = retrieve(query, top_k=8)
    context_blocks = []
    for i, item in enumerate(retrieved, start=1):
        context_blocks.append(
            f"[Sumber {i}: {item['filename']}, halaman {item['page']}] "
            f"{item['text'][:850]}"
        )
    context = "\n\n".join(context_blocks)[:6500]

    prompt = (
        "Berdasarkan HANYA konteks artikel di bawah, tulis sintesis awal research gap "
        "untuk topik Supply Chain Risk Management dalam bahasa Indonesia. "
        "Jelaskan (1) tema yang telah banyak dibahas, (2) keterbatasan atau pertanyaan yang belum terjawab "
        "yang benar-benar didukung konteks, (3) usulan fokus penelitian dan kontribusi yang bisa diuji. "
        "Jangan membuat referensi, nama penulis, angka, atau fakta yang tidak ada dalam konteks. "
        "Jika bukti tidak cukup, katakan bahwa bukti belum cukup. Sebutkan nama file dan halaman sumber bila tersedia. "
        "Beri label bahwa ini adalah draft yang perlu diverifikasi.\n\n"
        "KONTEKS:\n" + context
    )

    generated = local_generator(
        prompt,
        max_new_tokens=300,
        do_sample=False,
        truncation=True
    )[0]["generated_text"]

    local_report_path = os.path.join(OUT_DIR, "Analisis_Research_Gap_Generasi_Lokal.md")
    with open(local_report_path, "w", encoding="utf-8") as f:
        f.write("# Draft Sintesis Research Gap (Model Lokal, tanpa API key)\n\n")
        f.write("> Draft otomatis. Verifikasi semua klaim terhadap PDF dan file bukti sebelum dikumpulkan.\n\n")
        f.write(generated)
        f.write("\n\n## Potongan konteks yang dipakai\n\n")
        f.write(context)
    print("Draft sintesis lokal tersimpan:", local_report_path)
    print("\n--- Draft hasil model ---\n")
    print(generated)
else:
    print("Dilewati karena model lokal tidak berhasil dimuat. Gunakan laporan berbasis aturan dan lakukan sintesis manual.")

Dilewati karena model lokal tidak berhasil dimuat. Gunakan laporan berbasis aturan dan lakukan sintesis manual.


In [14]:
import os

for root, dirs, filenames in os.walk(OUT_DIR):
    for name in filenames:
        path = os.path.join(root, name)
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"{size_mb:10.2f} MB | {name}")

      0.01 MB | Literature_GAP_Matrix_Tanpa_API.csv
      0.04 MB | Bukti_Retrieval_Tanpa_API.csv
      0.00 MB | Ringkasan_Kata_Kunci_Gap.csv
      0.00 MB | Analisis_Research_Gap.md
      0.00 MB | Audit_Metadata_dan_Gap.csv
      0.05 MB | Literature_Gap_Matrix.csv
      0.29 MB | Bukti_Kutipan_dan_Halaman.csv
      0.00 MB | Usulan_Topik_Paper.md
      0.29 MB | Template_Validasi_Manual.csv
      0.00 MB | Ringkasan_Tema_Gap.csv


In [15]:
import os

zip_lama = os.path.join(
    OUT_DIR,
    "Hasil_RAG_Literature_Gap_Matrix.zip"
)

if os.path.exists(zip_lama):
    os.remove(zip_lama)
    print("ZIP lama berhasil dihapus.")
else:
    print("ZIP lama tidak ditemukan.")

ZIP lama tidak ditemukan.


In [16]:
import os
import zipfile
from google.colab import files

zip_path = "/content/Hasil_RAG_Literature_Gap_Matrix_Final.zip"

with zipfile.ZipFile(
    zip_path, "w", zipfile.ZIP_DEFLATED
) as zf:
    for root, dirs, filenames in os.walk(OUT_DIR):
        for filename in filenames:
            full_path = os.path.join(root, filename)

            # Abaikan file ZIP agar tidak ikut dikemas
            if filename.lower().endswith(".zip"):
                continue

            zf.write(
                full_path,
                arcname=os.path.relpath(full_path, OUT_DIR)
            )

print(
    "Ukuran ZIP:",
    round(os.path.getsize(zip_path) / (1024 * 1024), 2),
    "MB"
)

files.download(zip_path)

Ukuran ZIP: 0.19 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>